In [1]:
# STEP 1 — Import Libraries

import pandas as pd

file_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_wall_area_update.csv"

In [2]:
# STEP 2 — Load Dataset

df = pd.read_csv(file_path)

print("Dataset loaded.")
print(df.shape)
df.head()

Dataset loaded.
(117, 219)


,BUILDING_REFERENCE_NUMBER,OSG_REFERENCE_NUMBER,ADDRESS1,ADDRESS2,ADDRESS3,POSTCODE,LATITUDE,LONGITUDE,INSPECTION_DATE,TYPE_OF_ASSESSMENT,...,RIDGE_TO_ATTIC_HEIGHT,HALF_WALL_WIDTH,ROOF_WIDTH,HALF_ROOF_AREA,ROOF_AREA,USABLE_ROOF_LENGTH,USABLE_ROOF_WIDTH,PANELS_ALONG_LENGTH,PANELS_ALONG_WIDTH,POSSIBLE_PV_PANELS
0,1001829067,122009933.0,19 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.055692,-4.223337,9/29/25,"RdSAP, existing dwelling",...,2.291288,2.291288,3.240370,14.849242,29.698485,3.895189,2.754315,2,2,0
1,1001951858,122010025.0,GLENVIEW,FINTRY,GLASGOW,G63 0XJ,56.052824,-4.220640,9/19/25,"RdSAP, existing dwelling",...,3.240370,3.240370,4.582576,44.547727,89.095454,8.262944,3.895189,4,3,8
2,1001829071,122009868.0,22 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.055503,-4.223691,7/29/25,"RdSAP, existing dwelling",...,2.915476,2.915476,4.123106,24.041631,48.083261,4.956309,3.504640,2,3,2
3,1234761001,122009968.0,1 MENZIES TERRACE,FINTRY,GLASGOW,G63 0YJ,56.058427,-4.224838,7/22/25,"RdSAP, existing dwelling",...,3.253204,3.253204,4.600725,44.901281,89.802561,8.295669,3.910616,4,3,8
4,1001991633,122009892.0,50 MAIN STREET,FINTRY,GLASGOW,G63 0XF,56.054738,-4.228307,7/17/25,"RdSAP, existing dwelling",...,6.563332,4.163332,7.772429,97.077613,194.155227,10.616497,6.606565,5,5,21


In [3]:
# STEP 3 — Define Window Area Equations

window_coefficients = {
    "A": (0.1220, 6.875),
    "B": (0.1220, 6.875),
    "C": (0.1220, 6.875),
    "D": (0.1294, 5.515),
    "E": (0.1239, 7.332),
    "F": (0.1252, 5.520),
    "G": (0.1356, 5.242),
    "H": (0.0948, 6.534),
    "I": (0.1382, -0.027),
    "J": (0.1435, -0.403),
    "K": (0.1435, -0.403),
    "L": (0.1435, -0.403)
}

In [4]:
# STEP 4 — Define Glazing Adjustment Factors

glazing_scaling = {
    0: 1.0,
    1: 1.0,
    2: 1.20,
    4: 1.50
}

In [5]:
# STEP 5 — Calculate Typical Window Area

def calculate_typical_window_area(row):
    
    age_band = row["AGE_BAND"]
    tfa = row["TOTAL_FLOOR_AREA"]
    
    if age_band in window_coefficients:
        a, b = window_coefficients[age_band]
        return a * tfa + b
    else:
        return None

df["TYPICAL_WINDOW_AREA"] = df.apply(calculate_typical_window_area, axis=1)

df.head()

,BUILDING_REFERENCE_NUMBER,OSG_REFERENCE_NUMBER,ADDRESS1,ADDRESS2,ADDRESS3,POSTCODE,LATITUDE,LONGITUDE,INSPECTION_DATE,TYPE_OF_ASSESSMENT,...,RIDGE_TO_ATTIC_HEIGHT,HALF_WALL_WIDTH,ROOF_WIDTH,HALF_ROOF_AREA,ROOF_AREA,USABLE_ROOF_LENGTH,USABLE_ROOF_WIDTH,PANELS_ALONG_LENGTH,PANELS_ALONG_WIDTH,POSSIBLE_PV_PANELS
0,1001829067,122009933.0,19 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.055692,-4.223337,9/29/25,"RdSAP, existing dwelling",...,2.291288,2.291288,3.240370,14.849242,29.698485,3.895189,2.754315,2,2,0
1,1001951858,122010025.0,GLENVIEW,FINTRY,GLASGOW,G63 0XJ,56.052824,-4.220640,9/19/25,"RdSAP, existing dwelling",...,3.240370,3.240370,4.582576,44.547727,89.095454,8.262944,3.895189,4,3,8
2,1001829071,122009868.0,22 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.055503,-4.223691,7/29/25,"RdSAP, existing dwelling",...,2.915476,2.915476,4.123106,24.041631,48.083261,4.956309,3.504640,2,3,2
3,1234761001,122009968.0,1 MENZIES TERRACE,FINTRY,GLASGOW,G63 0YJ,56.058427,-4.224838,7/22/25,"RdSAP, existing dwelling",...,3.253204,3.253204,4.600725,44.901281,89.802561,8.295669,3.910616,4,3,8
4,1001991633,122009892.0,50 MAIN STREET,FINTRY,GLASGOW,G63 0XF,56.054738,-4.228307,7/17/25,"RdSAP, existing dwelling",...,6.563332,4.163332,7.772429,97.077613,194.155227,10.616497,6.606565,5,5,21


In [6]:
# STEP 6 — Apply Glazing Scaling

def adjust_window_area(row):
    
    base_area = row["TYPICAL_WINDOW_AREA"]
    glazing_class = row["GLAZED_AREA"]
    
    factor = glazing_scaling.get(glazing_class, 1.0)
    
    return base_area * factor

df["WINDOW_AREA"] = df.apply(adjust_window_area, axis=1)

In [7]:
# STEP 7 — Calculate Window-to-Wall Ratio

df["WWR"] = df["WINDOW_AREA"] / df["WALL_AREA"]

df[["WINDOW_AREA", "WWR"]].head()

,WINDOW_AREA,WWR
0,12.5358,0.189968
1,22.3276,0.143551
2,14.3142,NaN
3,23.0673,0.147722
4,19.3444,NaN


In [8]:
# STEP 8 — Save Updated Dataset

output_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_window_area_update.csv"

df.to_csv(output_path, index=False)

print("New dataset saved to:")
print(output_path)

New dataset saved to:
/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_window_area_update.csv
